# 第 10 週 實作｜面積與體積

期中考後轉入積分的應用。這一整段的心法只有一句:<strong>切成薄片、寫出一片、加起來、取極限</strong>。今天用它推出你國中背過的球體積公式。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜數值切片:看黎曼和收斂到真體積

觀念 3 用「切→近似→加起來→取極限」推出圓盤公式。這格把中間那步<strong>停在有限 n</strong>,看它怎麼一步步逼近解析解。


In [ ]:
# 旋轉 y = sqrt(x) 在 [0,4] 繞 x 軸,解析解 8*pi
f = np.sqrt                      # 用 np 版本,才能同時吃純量與陣列
a, b = 0.0, 4.0
exact = 8 * math.pi

def disk_riemann(n):
    """把旋轉體切成 n 個圓柱,加總體積(中點取樣)"""
    h = (b - a) / n
    return sum(math.pi * f(a + (i + 0.5)*h)**2 * h for i in range(n))

print(f"{'n':>6} {'圓柱和':>14} {'誤差':>12} {'誤差比值':>10}")
prev = None
for n in [4, 8, 16, 32, 64, 128]:
    v = disk_riemann(n)
    e = abs(v - exact)
    r = f"{prev/e:10.2f}" if prev and e > 0 else "         -"
    print(f"{n:6d} {v:14.8f} {e:12.3e} {r}")
    prev = e
print(f"{'解析解':>6} {exact:14.8f}")

# 畫出 n=8 的圓柱堆疊(側視圖)
n = 8
h = (b - a) / n
fig, ax = plt.subplots(figsize=(7, 3.5))
xs = np.linspace(a, b, 300)
ax.plot(xs, f(xs), 'C3', lw=2, label='y = sqrt(x)')
ax.plot(xs, -f(xs), 'C3', lw=2)
for i in range(n):
    xl, r = a + i*h, f(a + (i + 0.5)*h)
    ax.add_patch(plt.Rectangle((xl, -r), h, 2*r, fill=False, ec='C0', lw=0.8))
ax.set_title('n = 8 cylinders approximating the solid of revolution')
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# TODO 學生練習:把 f 換成 lambda t: t**2,區間 [0,1],解析解 pi/5
# 誤差比值還是 4 嗎?再試左端點取樣(把 (i+0.5) 改成 i),比值會變成多少?

## Lab 2｜圓盤 vs 殼層:同一題兩種算法

觀念 7 說兩種方法都對,差別在要不要反解。這格用 SymPy 把兩種都算一次,驗證答案相同,並比較式子的複雜度。


In [ ]:
x, y = sp.symbols('x y', nonnegative=True)

print("題目:y = x^2 在 [0,1],繞 y 軸旋轉\n")

# --- 殼層法:對 x 積,不必反解 ---
shell = sp.integrate(2*sp.pi*x * x**2, (x, 0, 1))
print("殼層法  V = ∫ 2*pi*x*(x^2) dx  from 0 to 1")
print("            =", shell, "=", float(shell))

# --- 圓盤法:對 y 積,必須反解 x = sqrt(y),而且是「大圓柱挖洞」---
disk = sp.integrate(sp.pi*(1**2 - (sp.sqrt(y))**2), (y, 0, 1))
print("\n圓盤法  先反解 x = sqrt(y),外半徑 1、內半徑 sqrt(y)")
print("        V = ∫ pi*(1^2 - y) dy  from 0 to 1")
print("            =", disk, "=", float(disk))
print("\n兩者相等?", sp.simplify(shell - disk) == 0)

# --- 一個圓盤法幾乎做不到的例子 ---
print("\n" + "="*56)
print("題目:y = exp(-x^2) 在 [0,1],繞 y 軸旋轉")
shell2 = sp.integrate(2*sp.pi*x*sp.exp(-x**2), (x, 0, 1))
print("殼層法  V =", sp.simplify(shell2), "=", float(shell2))
print("圓盤法  需要反解 x = sqrt(-ln y) —— 積分幾乎不可能做")
print("        →", sp.integrate(sp.pi*(1 - (-sp.log(y))), (y, sp.exp(-1), 1)),
      "(而且上下限也變得很麻煩)")

In [ ]:
# TODO 學生練習:y = x^3 在 [0,1] 繞 x 軸(不是 y 軸!)
# 這次哪一種比較好?兩種都算一次驗證答案相同(應為 pi/7)